# Random-Walk Metropolis-Hastings

This tutorial samples a posterior with the **Metropolis-Hastings** algorithm in Turing.jl, and tunes its proposal by hand.

Metropolis-Hastings proposes a new point and accepts it with a probability that depends on how the posterior density there compares with the density at the current point; a rejected proposal repeats the current point. Everything depends on the **proposal**. The panel's `MH()` proposes each parameter from its **prior**, independently of where the chain is. Here we use a **random walk** instead: the proposal is the current point plus a normal step, and the size of that step is the setting to tune.

In [ ]:
Pollis.packages("Turing", "MCMCChains", "StatsPlots")

In [ ]:
using Turing, MCMCChains, StatsPlots, LinearAlgebra, Random, Statistics

Random.seed!(42);  # fixed seed: the same data and draws on every run

## The Model

Fifty observations simulated from a normal distribution with mean 3 and standard deviation 1.5. We estimate both parameters, with a wide Normal(0, 10) prior on the mean and an Exponential(1) prior on the standard deviation.

In [ ]:
y = rand(Normal(3.0, 1.5), 50)

@model function normal_model(y)
    mu ~ Normal(0, 10)
    sigma ~ Exponential(1)
    for i in eachindex(y)
        y[i] ~ Normal(mu, sigma)
    end
end

model = normal_model(y);

Two helpers. A rejected proposal repeats the current point, so the share of iterations in which `mu` changes is the **acceptance rate**. The **effective sample size** (ESS) is the number of independent draws the chain is worth. `draw` runs a sampler and returns an MCMCChains chain, as the panel does.

In [ ]:
draw(sampler, n; warmup = 1_000) = sample(model, sampler, n; discard_initial = warmup, progress = false, chain_type = MCMCChains.Chains)

acceptance(chain) = mean(diff(vec(chain[:mu])) .!= 0)

function report(name, chain)
    e = ess(chain)
    ess_by = Dict(zip(e[:, :parameters], e[:, :ess]))
    println(rpad(name, 22), "acceptance ", rpad(round(acceptance(chain); digits = 3), 8),
            "ESS mu ", rpad(round(Int, ess_by[:mu]), 7), "ESS sigma ", round(Int, ess_by[:sigma]), "  of ", length(chain), " draws")
end;

## Proposals from the Prior

First what `MH()` with no arguments does: each proposal is drawn from the prior. The prior on `mu` is spread over tens of units, while fifty observations pin the posterior down to a fraction of a unit, so almost every proposal lands where the posterior density is negligible, and is rejected.

In [ ]:
c_prior = draw(MH(), 10_000)
report("Prior proposals", c_prior)

## A Random Walk

Given a covariance matrix, `MH(Sigma)` makes random-walk proposals: the current point plus a step drawn from a multivariate normal with mean zero and covariance `Sigma`. Turing takes these steps in the **unconstrained space**, where `sigma`, which must be positive, is replaced by `log(sigma)`.

We try three step sizes, `scale^2` times the identity matrix.

In [ ]:
scales = (0.01, 0.2, 3.0)
chains = Dict(s => draw(MH(s^2 * Matrix(1.0I, 2, 2)), 10_000) for s in scales)
for s in scales
    report("Scale $s", chains[s])
end

In [ ]:
plot([plot(vec(chains[s][:mu]); title = "scale $s", legend = false, xlabel = "iteration", ylabel = "mu") for s in scales]...;
     layout = (3, 1), size = (700, 650))

The three traces show the two ways a random walk fails:

- **Too small**: almost every proposal is accepted, but each moves the chain very little, so it wanders slowly and successive draws are almost the same.
- **Too large**: almost every proposal overshoots into the tails and is rejected, so the trace is made of long flat stretches.
- **In between**: the chain moves often and far, and the ESS is many times larger.

A classic guide (Roberts, Gelman and Gilks, 1997) is an acceptance rate of about 0.23 for random walks in many dimensions, and about 0.44 in one. With two parameters, we aim for about 0.35.

## Tuning the Scale

We tune the scale with short **pilot runs**: after each one, the scale grows if the acceptance rate was above the target and shrinks if it was below. The pilot draws are only for tuning and are thrown away. Changing the proposal during the run that we keep would break the Markov property the algorithm relies on, which is why the tuning happens before it.

In [ ]:
target = 0.35
scale  = 1.0
for round_ in 1:10
    pilot = draw(MH(scale^2 * Matrix(1.0I, 2, 2)), 1_000; warmup = 0)
    a = acceptance(pilot)
    global scale *= exp(2 * (a - target))  # larger steps when too many are accepted, smaller when too few
    println("round ", lpad(round_, 2), ": acceptance ", rpad(round(a; digits = 3), 6), " -> scale ", round(scale; digits = 3))
end

c_scale = draw(MH(scale^2 * Matrix(1.0I, 2, 2)), 10_000)
report("Tuned scale", c_scale)

## Tuning the Shape

One scale for both parameters ignores that the posterior may be wider in one direction than in another, or tilted when the parameters are correlated. A better proposal has the **shape of the posterior**: the covariance of the pilot draws, measured in the unconstrained space where the steps are taken (`mu` and `log(sigma)`), multiplied by 2.38^2 / d for d parameters, the optimal scaling of Roberts, Gelman and Gilks.

In [ ]:
pilot = draw(MH(scale^2 * Matrix(1.0I, 2, 2)), 5_000)
z     = hcat(vec(pilot[:mu]), log.(vec(pilot[:sigma])))  # pilot draws in the unconstrained space
Sigma = cov(z)
d     = size(z, 2)

c_cov = draw(MH(2.38^2 / d * Sigma), 10_000)
report("Tuned covariance", c_cov)

## Checking the Chain

Tuning makes the sampler efficient; it does not prove it has converged. The trace should look like stationary noise, the autocorrelations should fall to near zero within a few lags, R-hat should be close to 1 and the ESS large enough for what we report.

In [ ]:
plot(c_cov)

In [ ]:
autocorplot(c_cov)

In [ ]:
summarystats(c_cov)

## Against NUTS

The panel's default sampler, NUTS, uses gradients to make long moves, and needs no tuning by hand. The fair comparison is the **ESS per second**. With two parameters and a cheap likelihood, a well-tuned random walk can hold its own; as the number of parameters grows, its efficiency falls quickly and NUTS pulls ahead. The first NUTS run includes compilation: run the cell twice for a fair timing.

In [ ]:
t_mh   = @elapsed c_mh   = draw(MH(2.38^2 / d * Sigma), 10_000)
t_nuts = @elapsed c_nuts = sample(model, NUTS(), 2_000; progress = false, chain_type = MCMCChains.Chains)

for (name, c, t) in (("Random-walk MH", c_mh, t_mh), ("NUTS", c_nuts, t_nuts))
    e = ess(c)
    ess_mu = Dict(zip(e[:, :parameters], e[:, :ess]))[:mu]
    println(rpad(name, 16), "ESS mu ", rpad(round(Int, ess_mu), 7), "seconds ", rpad(round(t; digits = 2), 7), "ESS per second ", round(Int, ess_mu / t))
end

## Summary

- `MH()` proposes from the prior, which works only when the posterior is close to the prior.
- `MH(Sigma)` makes random-walk proposals in the unconstrained space; the step size decides the efficiency.
- Tune with pilot runs, towards an acceptance rate of about 0.23 in many dimensions, 0.44 in one; then give the proposal the shape of the posterior, the pilot covariance times 2.38^2 / d.
- Discard the pilot draws, and check the final chain with traces, autocorrelations, R-hat and ESS.
- Compare samplers by ESS per second.